Generalised profiling tutorial

Nonlinear mass action

Alexander Johnston
Queensland University of Technology
a44.johnston@qut.edu.au


In [9]:
using Plots, DelimitedFiles, LeastSquaresOptim, DifferentialEquations, SparseArrays, NonlinearSolve, NLsolve, Random, Distributions, NLopt, Dierckx, LaTeXStrings, BSplineKit, Plots.PlotMeasures, LinearAlgebra
gr()

plot_font = "Computer Modern"
default(fontfamily=plot_font,linewidth=1,framestyle=:box,label=nothing,grid=true)

#Model parameters

r1 = 100;
r2 = 100;
K1 = 210;
K2 = 210;
c1_0 = 1000;
c2_0 = 400;

a = [r1, r2, K1, K2];

#Initial conditions
x0 = c1_0;
y0 = c2_0;

#Number of synthetic data points
N_data = 41;

#Ratio of number of points in the spline grid to the number of points in the synthetic data set 
grid_ratio = 10;

#Number of B-Splines
df = N_data - 1;

#Discretisation grid size for spline
N_t = grid_ratio*df;

t_start = 0;
t_end = 25;
tt = LinRange(t_start, t_end, N_t);

t = [0]
for (i, t_i) in enumerate(tt)
    if i%grid_ratio == 0
        t = vcat(t, t_i)
    end
end

#Standard deviation of additive Gaussian noise used to generate the synthetic data
sigma = 5; 


function NLLS_function(A, b)
    function matrix_equation(x)
    	A*x - b
    end
    x0 = ones(length(A[1,:]))
    result = LeastSquaresOptim.optimize(matrix_equation, x0, LeastSquaresOptim.Dogleg())
    return result.minimizer
end 
    


NLLS_function (generic function with 1 method)

Creating synthetic data sets.

Here we define the necessary functions to run the generalised profiling method. This involves constructing a B-Spline matrix and writing a function for the generalised log-likelihood.

In [10]:
#Create synthetic data for the nonlinear mass action model.

#Michaelis-Menten equations
function Michaelis_Menten!(du,u,a,t)
r1 = a[1];
r2 = a[2];
K1 = a[3];
K2 = a[4];
du[1]=-r1*u[1]/(K1 + u[1]); 
du[2]=r1*u[1]/(K1 + u[1]) - r2*u[2]/(K2 + u[2]);
end

function odesolver(t,ic,a)
tspan=(0.0,maximum(t))
prob=ODEProblem(Michaelis_Menten!,ic,tspan, a)
alg=Tsit5()
sol=solve(prob,alg,saveat=t);
return sol
end

ic=[x0, y0]
sol = odesolver(tt,ic,a)
    
x = sol[1,:];
y = sol[2,:];

t = [0]
for (i, t_i) in enumerate(tt)
    if i%grid_ratio == 0
        t = vcat(t, t_i)
    end
end
        
sol_true = sol = odesolver(tt,ic,a)
x_true = sol[1,:];
y_true = sol[2,:];

#Create stochastic data using additive Gaussian noise applied at each data point for the true solution

dist=Normal(0,sigma);

x_model_subset = [x_true[1]];
y_model_subset = [y_true[1]];
for (i, t_i) in enumerate(tt)
    if i%grid_ratio == 0
        x_model_subset = vcat(x_model_subset, x_true[i])
        y_model_subset = vcat(y_model_subset, y_true[i])
    end
end

x_data_sample = zeros(length(x_model_subset));
y_data_sample = zeros(length(y_model_subset));

for (i, x_model_i) in enumerate(x_model_subset)
    x_data_sample[i] = x_model_i + rand(dist)
end

for (i, y_model_i) in enumerate(y_model_subset)
    y_data_sample[i] = y_model_i + rand(dist)
end


Loading the synthetic data set used in the paper "Efficient inference for differential equation models without numerical solvers" 

In [11]:
#The following sets are the specific synthetic data sets used to produce plots in the paper

t_data = readdlm("t_synthetic_data_Michaelis-Menten");
t_model = readdlm("t_synthetic_model_Michaelis-Menten");

x_data = readdlm("c1_synthetic_data_sample_Michaelis-Menten");
y_data = readdlm("c2_synthetic_data_sample_Michaelis-Menten");

x_model = readdlm("c1_synthetic_model_Michaelis-Menten");
y_model = readdlm("c2_synthetic_model_Michaelis-Menten");


In [12]:
#Construct a B-Spline matrix

spline_order = BSplineKit.BSplineOrder(4)  # This corresponds to cubic splines
B = BSplineKit.BSplineBasis(BSplineKit.BSplineOrder(4), LinRange(t_start, t_end, df-1))
B = RecombinedBSplineBasis(B, BSplineKit.Natural())

B_matrix = collocation_matrix(B, tt, BSplineKit.Derivative(0), SparseMatrixCSC{Float64})
B_matrix[:,1] = ones(N_t)


#Construct a vector of indices, i_obs, showing the indices of the fine grid of true data, tt, that correspond to values of the synthetic data subset, t

i_obs_x = []
for (i, t_i) in enumerate(t_model)
    if t_i in t_data
        append!(i_obs_x,i)
    end
end

i_obs_y = []
for (i, t_i) in enumerate(t_model)
    if t_i in t_data
        append!(i_obs_y,i)
    end
end


#Construct an A_obs matrix. This matrix constructs a matrix in which each row consists of zeros and a single one. The location of the single one is specificed by the index of the i_obs vector.

t_model = collect(t_model)

function create_A_obs(i_obs)
    A_obs = zeros(Int, length(i_obs), length(t_model))
    for i in range(1, length(i_obs))
        A_obs[i, i_obs[i]] = 1
    end
    A_obs
end

#Construct A_obs, B_obs, and beta_est

 xy_data = vcat(x_data, y_data);
 xy_model = vcat(x_model, y_model);

 A_obs_x = create_A_obs(i_obs_x);
 A_obs_y = create_A_obs(i_obs_y);
 A_obs = A_obs_x;
 B_obs_x = A_obs_x*B_matrix;
 B_obs_y = A_obs_y*B_matrix;
 B_obs = B_obs_x;
 beta_x = NLLS_function(B_obs./sigma, vec(x_data)./sigma);
 beta_y = NLLS_function(B_obs./sigma, vec(y_data)./sigma);

#Construct first and second derivative matrix approximations

N = length(x_model)
dt = t_model[end]/(N-1)
D1 = diagm(-1 => -1*ones(N-1), 0 => 0*ones(N-1), 1 => ones(N-1))
D1 = (1 / (2*dt))*D1[1:N, :];
D1[1, 1] = -1/dt;
D1[1, 2] = 1/dt;
D1[N, N-1] = -1/dt;
D1[N, N] = 1/dt;

D1B = D1*B_matrix;

sigma_D = sqrt(2)*sigma/(t_data[2]-t_data[1]);

#Loglikelihood functions

function loglhood_sum(w,x_data,y_data,model_x,model_y,a,sigma)
    B_obs_aug_x = vcat(B_obs_x, w*dt*D1B);
    B_obs_aug_y = vcat(B_obs_y, w*dt*D1B);
    
    RHS_term_x = -a[1]*model_x./(a[3] .+ model_x);
    RHS_term_y = a[1]*model_x./(a[3] .+ model_x) - a[2]*model_y./(a[4] .+ model_y);
   
    x_data_aug = vcat(x_data./sigma, w*dt*RHS_term_x./sigma_D);
    y_data_aug = vcat(y_data./sigma, w*dt*RHS_term_y./sigma_D);    

    function f(x)
        [(x[1]+3)*(x[2]^3-7)+18,
        sin(x[2]*exp(x[1])-1)]
    end

    sol = nlsolve(f, [ 0.1, 1.2])
    sol.zero
    
    betas_x = NLLS_function(B_obs./sigma, vec(x_data)./sigma);
    betas_y = NLLS_function(B_obs./sigma, vec(y_data)./sigma);
    
    model_x = B_matrix*betas_x;
    model_y = B_matrix*betas_y;
    
    g=vcat(A_obs_x*B_matrix*betas_x, A_obs_y*B_matrix*betas_y);
    y1 = vcat(x_data, y_data);
    dist=Normal(0,sigma);  

    Dx = D1B*betas_x;
    Dy = D1B*betas_y;

    y2= vcat(Dx, Dy);
    
    f = vcat(-a[1]*model_x./(a[3] .+ model_x), a[1]*model_x./(a[3] .+ model_x) - a[2]*model_y./(a[4] .+ model_y));   
    
    e3=loglikelihood(dist, (y1.-g));
    e4 = sum(e3) - w*norm((y2 - f))^2/sigma_D^2; 
    return e4

end;

#New values of w based on standard deviations of results from previous step

function standard_deviation_data_norms(x_data,y_data,betas_x,betas_y,a,sigma)
    g=vcat(A_obs_x*B_matrix*betas_x, A_obs_y*B_matrix*betas_y);
    y = vcat(x_data, y_data);
    dist=Normal(0,sigma);  
    
    e=std(y.-g);
    return e
end;

function standard_deviation_model_norms(betas_x,betas_y,a,sigma)
    Dx = D1B*betas_x;
    Dy = D1B*betas_y;
                    
    y2= vcat(Dx, Dy);

    model_x = B_matrix*betas_x;
    model_y = B_matrix*betas_y;
    
    f = vcat(-a[1]*model_x./(a[3] .+ model_x), a[1]*model_x./(a[3] .+ model_x) - a[2]*model_y./(a[4] .+ model_y));   
                          
    e=std(y2 - f);
    return e
end;

#Optimising the likelihood

function Optimise(fun,θ₀,lb,ub)    
    tomax=(θ,∂θ)->fun(θ)
    opt=Opt(:LN_NELDERMEAD,length(θ₀))
    opt.max_objective=tomax
    opt.lower_bounds=lb      
    opt.upper_bounds=ub
    opt.maxtime=1*60
    res = NLopt.optimize(opt,θ₀)
    return res[[2,1]];
end

function funmle_sum(a)
return loglhood_sum(w,x_data,y_data,model_x,model_y,a,sigma);
end



LoadError: MethodError: no method matching BSplineBasis(::Natural)

[0mClosest candidates are:
[0m  BSplineBasis([91m::BSplineOrder{k}[39m, [91m::AbstractVector{T}[39m; augment) where {k, T}
[0m[90m   @[39m [35mBSplineKit[39m [90m~/.julia/packages/BSplineKit/FOLdw/src/BSplines/[39m[90m[4mbasis.jl:74[24m[39m
[0m  BSplineBasis([91m::BSplineOrder[39m, [91m::Tuple[39m; kws...)
[0m[90m   @[39m [35mBSplineKit[39m [90m~/.julia/packages/BSplineKit/FOLdw/src/BSplines/[39m[90m[4mbasis.jl:92[24m[39m
[0m  BSplineBasis([91m::Integer[39m, Any...; kwargs...)
[0m[90m   @[39m [35mBSplineKit[39m [90m~/.julia/packages/BSplineKit/FOLdw/src/BSplines/[39m[90m[4mbasis.jl:101[24m[39m


Intial data match - Iteration 0

In [ ]:
#Sum analysis

r1min= 10;
r1max= 150;

r2min= 10;
r2max= 150;

K1min= 100;
K1max= 300;

K2min= 100;
K2max= 400;

#Initial regularisation parameter

w = 0;

r1_test = 110;
r2_test = 90;
K1_test = 150;
K2_test = 150;

a_test = [r1_test, r2_test, K1_test, K2_test];
a = a_test;

x_model_test = 0.5*ones(N);
y_model_test = 0.5*ones(N);

model_x = x_model_test;
model_y = y_model_test;

θG=a_test;
lb=[r1min, r2min, K1min, K2min];
ub=[r1max, r2max, K1max, K2max];

#Sum analysis

(xopt_sum,fopt_sum)=Optimise(funmle_sum,θG,lb,ub)
r1mle_sum=xopt_sum[1]
r2mle_sum=xopt_sum[2]
K1mle_sum=xopt_sum[3]
K2mle_sum=xopt_sum[4]

#Solving the linear algebra optimisation problem

B_obs_aug_x = vcat(B_obs_x./sigma, w*dt*D1B./sigma_D);
B_obs_aug_y = vcat(B_obs_y./sigma, w*dt*D1B./sigma_D);

RHS_term_x = -a[1]*model_x./(a[3] .+ model_x);
RHS_term_y = a[1]*model_x./(a[3] .+ model_x) - a[2]*model_y./(a[4] .+ model_y);

#RHS_term_x = -a[1]*model_x;
#RHS_term_y = a[1]*model_x. - a[2]*model_y;

x_data_aug = vcat(x_data./sigma, w*dt*RHS_term_x./sigma_D);
y_data_aug = vcat(y_data./sigma, w*dt*RHS_term_y./sigma_D);

betas_x_sum = NLLS_function(B_obs_aug_x, vec(x_data_aug));
betas_y_sum = NLLS_function(B_obs_aug_y, vec(y_data_aug));

Dx = D1B*betas_x_sum;
Dy = D1B*betas_y_sum;

y2= vcat(Dx, Dy);

f = vcat(-a[1]*model_x./(a[3] .+ model_x), a[1]*model_x./(a[3] .+ model_x) - a[2]*model_y./(a[4] .+ model_y));    

B_matrix_betas_x_sum = B_matrix*betas_x_sum;
B_matrix_betas_y_sum = B_matrix*betas_y_sum;



In [ ]:
#Plot Initial Data Match

p1a0=scatter(t_data,x_data,mc=:blue,msc=:blue,label=false,msw=0,ms=4, dpi = 1000);
p1a0=scatter!(t_data,y_data,mc=:green,msc=:green,label=false,msw=0,ms=4);
p1a0=plot!(t_model, B_matrix_betas_x_sum,lw=2,xlabel=L"t",color=:blue,label=false);
p1a0=plot!(t_model, B_matrix_betas_y_sum,lw=2,xlabel=L"t",color=:green,label=false);
p1a0=plot!(xticks=([0, 5, 10, 15],[L"0",L"5", L"10", L"15"]));
p1a0=plot!(yticks=([0, 1, 2, 3],[L"0",L"1", L"2", L"3"]));
p1a0=plot!(xlabel=L"t", ylabel = L"a(t), b(t)",color=:red, xlims=(t_data[1]-1,t_data[end]+1), ylims=(0,c1_0));
p1a0=plot!(xguidefontsize=10, yguidefontsize=10,xtickfontsize=10, ytickfontsize=10, label = false, size = (400,300))
title!("Initial Data Match", titlefontsize=12)

println("w^(0) = "  * string(w))
plot(p1a0)


Iteration 1

In [ ]:
#New weight and parameters

w = 10^(-1);

θG=a_test;
(xopt_sum,fopt_sum)=Optimise(funmle_sum,θG,lb,ub)
r1mle_sum=xopt_sum[1]
r2mle_sum=xopt_sum[2]
K1mle_sum=xopt_sum[3]
K2mle_sum=xopt_sum[4]

a_test = [r1mle_sum, r2mle_sum, K1mle_sum, K2mle_sum];
a = a_test;
betas_x = betas_x_sum;
betas_y = betas_y_sum;
model_x = B_matrix*betas_x;
model_y = B_matrix*betas_y;

#Solving the linear algebra optimisation problem

B_obs_aug_x_1 = vcat(B_obs_x./sigma, w*dt*D1B./sigma_D);
B_obs_aug_y_1 = vcat(B_obs_y./sigma, w*dt*D1B./sigma_D);

RHS_term_x = -a[1]*model_x./(a[3] .+ model_x);
RHS_term_y = a[1]*model_x./(a[3] .+ model_x) - a[2]*model_y./(a[4] .+ model_y);

x_data_aug = vcat(x_data./sigma, w*dt*RHS_term_x./sigma_D);
y_data_aug = vcat(y_data./sigma, w*dt*RHS_term_y./sigma_D);

betas_x_sum_1 = NLLS_function(B_obs_aug_x_1, vec(x_data_aug));
betas_y_sum_1 = NLLS_function(B_obs_aug_y_1, vec(y_data_aug));

Dx_1 = D1B*betas_x_sum_1;
Dy_1 = D1B*betas_y_sum_1;

y2= vcat(Dx_1, Dy_1);

f = vcat(-a[1]*model_x./(a[3] .+ model_x), a[1]*model_x./(a[3] .+ model_x) - a[2]*model_y./(a[4] .+ model_y));   

B_matrix_betas_x_sum_1 = B_matrix*betas_x_sum_1;
B_matrix_betas_y_sum_1 = B_matrix*betas_y_sum_1;


In [ ]:
#Plot Iteration 1
p1a1=scatter(t_data,x_data,mc=:blue,msc=:blue,label=false,msw=0,ms=4, dpi = 1000);
p1a1=scatter!(t_data,y_data,mc=:green,msc=:green,label=false,msw=0,ms=4);
p1a1=plot!(t_model, B_matrix_betas_x_sum_1,lw=2,xlabel=L"t",color=:blue,label=false);
p1a1=plot!(t_model, B_matrix_betas_y_sum_1,lw=2,xlabel=L"t",color=:green,label=false);
p1a1=plot!(xticks=([0, 5, 10, 15],[L"0",L"5", L"10", L"15"]));
p1a1=plot!(yticks=([0, 1, 2, 3],[L"0",L"1", L"2", L"3"]));
p1a1=plot!(xlabel=L"t", ylabel = L"a(t), b(t)",color=:red, xlims=(t_data[1]-1,t_data[end]+1), ylims=(0,c1_0));
p1a1=plot!(xguidefontsize=10, yguidefontsize=10,xtickfontsize=10, ytickfontsize=10, label = false, size = (400,300))
title!("Iteration 1", titlefontsize=12)

println("w^(1) = "  * string(w))
plot(p1a1)

Iteration 2

In [ ]:
#New weight and parameters
θG=a_test;
(xopt_sum,fopt_sum)=Optimise(funmle_sum,θG,lb,ub)
r1mle_sum=xopt_sum[1]
r2mle_sum=xopt_sum[2]
K1mle_sum=xopt_sum[3]
K2mle_sum=xopt_sum[4]

a_test = [r1mle_sum, r2mle_sum, K1mle_sum, K2mle_sum];
a = a_test;
betas_x = betas_x_sum_1;
betas_y = betas_y_sum_1;
model_x = B_matrix*betas_x;
model_y = B_matrix*betas_y;

w = (standard_deviation_data_norms(x_data, y_data, betas_x, betas_y, a_test, sigma))/(standard_deviation_model_norms(betas_x, betas_y, a_test, sigma))*sigma_D/sigma*(2*N_data-1)/(2*length(tt)-1);

#Solving the linear algebra optimisation problem

B_obs_aug_x_2 = vcat(B_obs_x./sigma, w*dt*D1B./sigma_D);
B_obs_aug_y_2 = vcat(B_obs_y./sigma, w*dt*D1B./sigma_D);

RHS_term_x = -a[1]*model_x./(a[3] .+ model_x);
RHS_term_y = a[1]*model_x./(a[3] .+ model_x) - a[2]*model_y./(a[4] .+ model_y);

x_data_aug = vcat(x_data./sigma, w*dt*RHS_term_x./sigma_D);
y_data_aug = vcat(y_data./sigma, w*dt*RHS_term_y./sigma_D);

betas_x_sum_2 = NLLS_function(B_obs_aug_x_2, vec(x_data_aug));
betas_y_sum_2 = NLLS_function(B_obs_aug_y_2, vec(y_data_aug));

Dx_2 = D1B*betas_x_sum_2;
Dy_2 = D1B*betas_y_sum_2;

y2= vcat(Dx_2, Dy_2);

f = vcat(-a[1]*model_x./(a[3] .+ model_x), a[1]*model_x./(a[3] .+ model_x) - a[2]*model_y./(a[4] .+ model_y));   

B_matrix_betas_x_sum_2 = B_matrix*betas_x_sum_2;
B_matrix_betas_y_sum_2 = B_matrix*betas_y_sum_2;


In [ ]:
#Plot Iteration 2
p1a2=scatter(t_data,x_data,mc=:blue,msc=:blue,label=false,msw=0,ms=4, dpi = 1000);
p1a2=scatter!(t_data,y_data,mc=:green,msc=:green,label=false,msw=0,ms=4);
p1a2=plot!(t_model, B_matrix_betas_x_sum_2,lw=2,xlabel=L"t",color=:blue,label=false);
p1a2=plot!(t_model, B_matrix_betas_y_sum_2,lw=2,xlabel=L"t",color=:green,label=false);
p1a2=plot!(xticks=([0, 5, 10, 15],[L"0",L"5", L"10", L"15"]));
p1a2=plot!(yticks=([0, 1, 2, 3],[L"0",L"1", L"2", L"3"]));
p1a2=plot!(xlabel=L"t", ylabel = L"a(t), b(t)",color=:red, xlims=(t_data[1]-1,t_data[end]+1), ylims=(0,c1_0));
p1a2=plot!(xguidefontsize=10, yguidefontsize=10,xtickfontsize=10, ytickfontsize=10, label = false, size = (400,300))
title!("Iteration 2", titlefontsize=12)

println("w^(2) = "  * string(w))
plot(p1a2)

Iteration 3

In [ ]:
#New weight and parameters
θG=a_test;
(xopt_sum,fopt_sum)=Optimise(funmle_sum,θG,lb,ub)
r1mle_sum=xopt_sum[1]
r2mle_sum=xopt_sum[2]
K1mle_sum=xopt_sum[3]
K2mle_sum=xopt_sum[4]

a_test = [r1mle_sum, r2mle_sum, K1mle_sum, K2mle_sum];
a = a_test;
betas_x = betas_x_sum_2;
betas_y = betas_y_sum_2;
model_x = B_matrix*betas_x;
model_y = B_matrix*betas_y;

w = (standard_deviation_data_norms(x_data, y_data, betas_x, betas_y, a_test, sigma))/(standard_deviation_model_norms(betas_x, betas_y, a_test, sigma))*sigma_D/sigma*(2*N_data-1)/(2*length(tt)-1);

#Solving the linear algebra optimisation problem

B_obs_aug_x_3 = vcat(B_obs_x./sigma, w*dt*D1B./sigma_D);
B_obs_aug_y_3 = vcat(B_obs_y./sigma, w*dt*D1B./sigma_D);

RHS_term_x = -a[1]*model_x./(a[3] .+ model_x);
RHS_term_y = a[1]*model_x./(a[3] .+ model_x) - a[2]*model_y./(a[4] .+ model_y);

x_data_aug = vcat(x_data./sigma, w*dt*RHS_term_x./sigma_D);
y_data_aug = vcat(y_data./sigma, w*dt*RHS_term_y./sigma_D);

betas_x_sum_3 = NLLS_function(B_obs_aug_x_3, vec(x_data_aug));
betas_y_sum_3 = NLLS_function(B_obs_aug_y_3, vec(y_data_aug));

Dx_3 = D1B*betas_x_sum_3;
Dy_3 = D1B*betas_y_sum_3;

y2= vcat(Dx_3, Dy_3);

f = vcat(-a[1]*model_x./(a[3] .+ model_x), a[1]*model_x./(a[3] .+ model_x) - a[2]*model_y./(a[4] .+ model_y));   

B_matrix_betas_x_sum_3 = B_matrix*betas_x_sum_3;
B_matrix_betas_y_sum_3 = B_matrix*betas_y_sum_3;

betas_x = betas_x_sum_3;
betas_y = betas_y_sum_3;
model_x = B_matrix*betas_x;
model_y = B_matrix*betas_y;



In [ ]:
#Plot Iteration 3
p1a3=scatter(t_data,x_data,mc=:blue,msc=:blue,label=false,msw=0,ms=4, dpi = 1000);
p1a3=scatter!(t_data,y_data,mc=:green,msc=:green,label=false,msw=0,ms=4);
p1a3=plot!(t_model, B_matrix_betas_x_sum_3,lw=2,xlabel=L"t",color=:blue,label=false);
p1a3=plot!(t_model, B_matrix_betas_y_sum_3,lw=2,xlabel=L"t",color=:green,label=false);
p1a3=plot!(xticks=([0, 5, 10, 15],[L"0",L"5", L"10", L"15"]));
p1a3=plot!(yticks=([0, 1, 2, 3],[L"0",L"1", L"2", L"3"]));
p1a3=plot!(xlabel=L"t", ylabel = L"a(t), b(t)",color=:red, xlims=(t_data[1]-1,t_data[end]+1), ylims=(0,c1_0));
p1a3=plot!(xguidefontsize=10, yguidefontsize=10,xtickfontsize=10, ytickfontsize=10, label = false, size = (400,300))
title!("Iteration 3", titlefontsize=12)

println("w^(3) = "  * string(w))
plot(p1a3)


In [ ]:
#Profile for r1

import Base: +, -, *

function univariater1(r1)
    a=zeros(3)    
    function funr1(a)
    return loglhood_sum(w,x_data, y_data, model_x, model_y, [r1,a[1],a[2],a[3]], sigma)
    end
    θG=[r2mle_sum, K1mle_sum, K2mle_sum]
    lb=[r2min, K1min, K2min] 
    ub=[r2max, K1max, K2max] 
    (xopt,fopt)=Optimise(funr1,θG,lb,ub)
    return fopt,xopt
    end 

#Take a grid of M points to plot the univariate profile likelihood
M=50;
r1range=LinRange(80,125,M)
ff=zeros(M)
for i in 1:M
    ff[i]=univariater1(r1range[i])[1]
end



In [ ]:
sp3=Spline1D(r1range,ff.-maximum(ff),w=ones(length(r1range)),k=3,bc="nearest",s=1/100);
yy=Dierckx.evaluate(sp3,r1range);
r1_plot=plot(r1range,yy,lw=4,lc=:red,ylims=(-1,0.025),xlims=(r1range[1],r1range[end])size = (300, 300));
r1_plot=plot(r1range,ff.-maximum(ff),lw=4,lc=:red,xlims=(r1range[1],r1range[end]),size = (300, 300));
r1_plot=vline!([r1mle_sum],legend=false,xlabel=L"r_{1}",lw=3,color=:blue);
r1_plot=vline!([r1],legend=false,xlabel=L"r_{1}",lw=3,color=:green);
r1_plot=plot!(xlims=(80,125));
r1_plot=plot!(ylims=(-1,0.025),yticks=([-1,-0.5,0],[L"-1.0", L"-0.5", L"0.0"]));
r1_plot=plot!(ylabel = L"\bar{\ell}_{p}", xguidefontsize=14, yguidefontsize=14,xtickfontsize=14, ytickfontsize=14)

In [ ]:
#Profile for r2

function univariater2(r2)
    a=zeros(3)    
    function funr2(a)
    return loglhood_sum(w,x_data, y_data, model_x, model_y, [a[1],r2,a[2],a[3]], sigma)
    end
    θG=[r1mle_sum, K1mle_sum, K2mle_sum]
    lb=[r1min, K1min, K2min] 
    ub=[r1max, K1max, K2max] 
    (xopt,fopt)=Optimise(funr2,θG,lb,ub)
    return fopt,xopt
    end 

#Take a grid of M points to plot the univariate profile likelihood
M=50;
r2range=LinRange(80,125,M)
ff=zeros(M)
for i in 1:M
    ff[i]=univariater2(r2range[i])[1]
end



In [ ]:
sp3=Spline1D(r2range,ff.-maximum(ff),w=ones(length(r2range)),k=3,bc="nearest",s=1/100);
yy=Dierckx.evaluate(sp3,r2range);
r2_plot=plot(r2range,yy,lw=4,lc=:red,ylims=(-1,0.025),xlims=(r2range[1],r2range[end])size = (300, 300));
r2_plot=plot(r2range,ff.-maximum(ff),lw=4,lc=:red,xlims=(r2range[1],r2range[end]),size = (300, 300));
r2_plot=vline!([r2mle_sum],legend=false,xlabel=L"r_{2}",lw=3,color=:blue);
r2_plot=vline!([r2],legend=false,xlabel=L"r_{2}",lw=3,color=:green);
r2_plot=plot!(xlims=(80,125));
r2_plot=plot!(ylims=(-1,0.025),yticks=([-1,-0.5,0],[L"-1.0", L"-0.5", L"0.0"]));
r2_plot=plot!(ylabel = L"\bar{\ell}_{p}", xguidefontsize=14, yguidefontsize=14,xtickfontsize=14, ytickfontsize=14)

In [ ]:
#Profile for K1

function univariateK1(K1)
    a=zeros(3)    
    function funK1(a)
    return loglhood_sum(w,x_data, y_data, model_x, model_y, [a[1],a[2],K1,a[3]], sigma)
    end
    θG=[r1mle_sum, r2mle_sum, K2mle_sum]
    lb=[r1min, r2min, K2min] 
    ub=[r1max, r2max, K2max] 
    (xopt,fopt)=Optimise(funK1,θG,lb,ub)
    return fopt,xopt
    end 

#Take a grid of M points to plot the univariate profile likelihood
M=50;
K1range=LinRange(100,300,M)
ff=zeros(M)
for i in 1:M
    ff[i]=univariateK1(K1range[i])[1]
end


In [ ]:
sp3=Spline1D(K1range,ff.-maximum(ff),w=ones(length(K1range)),k=3,bc="nearest",s=1/100);
yy=Dierckx.evaluate(sp3,K1range);
K1_plot=plot(K1range,yy,lw=4,lc=:red,ylims=(-1,0.025),xlims=(K1range[1],K1range[end])size = (300, 300));
K1_plot=plot(K1range,ff.-maximum(ff),lw=4,lc=:red,xlims=(K1range[1],K1range[end]),size = (300, 300));
K1_plot=vline!([K1mle_sum],legend=false,xlabel=L"K_{1}",lw=3,color=:blue);
K1_plot=vline!([K1],legend=false,xlabel=L"K_{1}",lw=3,color=:green);
K1_plot=plot!(xlims=(K1min,K1max));
K1_plot=plot!(ylims=(-1,0.025),yticks=([-1,-0.5,0],[L"-1.0", L"-0.5", L"0.0"]));
K1_plot=plot!(ylabel = L"\bar{\ell}_{p}", xguidefontsize=14, yguidefontsize=14,xtickfontsize=14, ytickfontsize=14)

In [ ]:
#Profile for K2

function univariateK2(K2)
    a=zeros(3)    
    function funK2(a)
    return loglhood_sum(w,x_data, y_data, model_x, model_y, [a[1],a[2],a[3],K2], sigma)
    end
    θG=[r1mle_sum, r2mle_sum, K1mle_sum]
    lb=[r1min, r2min, K1min] 
    ub=[r1max, r2max, K1max] 
    (xopt,fopt)=Optimise(funK2,θG,lb,ub)
    return fopt,xopt
    end 

#Take a grid of M points to plot the univariate profile likelihood
M=50;
K2range=LinRange(100,400,M)
ff=zeros(M)
for i in 1:M
    ff[i]=univariateK2(K2range[i])[1]
end


In [ ]:
sp3=Spline1D(K2range,ff.-maximum(ff),w=ones(length(K2range)),k=3,bc="nearest",s=1/100);
yy=Dierckx.evaluate(sp3,K1range);
K2_plot=plot(K2range,yy,lw=4,lc=:red,ylims=(-1,0.025),xlims=(K2range[1],K2range[end])size = (300, 300));
K2_plot=plot(K2range,ff.-maximum(ff),lw=4,lc=:red,xlims=(K2range[1],K2range[end]),size = (300, 300));
K2_plot=vline!([K2mle_sum],legend=false,xlabel=L"K_{2}",lw=3,color=:blue);
K2_plot=vline!([K2],legend=false,xlabel=L"K_{2}",lw=3,color=:green);
K2_plot=plot!(xlims=(K2min,K2max));
K2_plot=plot!(ylims=(-1,0.025),yticks=([-1,-0.5,0],[L"-1.0", L"-0.5", L"0.0"]));
K2_plot=plot!(ylabel = L"\bar{\ell}_{p}", xguidefontsize=14, yguidefontsize=14,xtickfontsize=14, ytickfontsize=14)

In [ ]:
#Plot slice figure

slice_plot = plot(r1_plot, r2_plot, K1_plot, K2_plot, layout = grid(2, 2), size = (600, 350), dpi = 1000, right_margin=3mm, bottom_margin=3mm) 
plot(slice_plot)

In [ ]:
ic=[x0, y0]
sol = odesolver(tt,ic,[r1mle_sum,r2mle_sum,K1mle_sum,K2mle_sum])
    
x_predicted = sol[1,:];
y_predicted  = sol[2,:];

#Plot Iteration 3
p3b=scatter(t_data,x_data,mc=:blue,msc=:blue,label=false,msw=0,ms=4, dpi = 1000);
p3b=plot!(t_model, x_predicted,lw=2,xlabel=L"t",color=:blue,label=false);
p3b=scatter!(t_data,y_data,mc=:green,msc=:green,label=false,msw=0,ms=4);
p3b=plot!(t_model, y_predicted,lw=2,xlabel=L"t",color=:green,label=false);
#p3b=plot!(xticks=([0, 5, 10, 15],[L"0",L"5", L"10", L"15"]));
#p3b=plot!(yticks=([0, 1, 2, 3],[L"0",L"1", L"2", L"3"]));
p3b=plot!(xlabel=L"t", ylabel = L"c_{1}(t), c_{2}(t)",color=:red, xlims=(t_data[1]-1,t_data[end]+1), fontsize = 25);
p3b=plot!(xguidefontsize=20, yguidefontsize=20,xtickfontsize=20, ytickfontsize=20, label = false, size = (600,400))

plot(p3b)